# Criação de composição temporal para cubos de dados

Pegar o pacote Smosaic (https://github.com/brazil-data-cube/smosaic) desenvolvido pelo Gabriel e adapta-lo para ter essa função que ele quer. No caso a função é pegar o menor valor do periodo de composição, podendo ser generalizada para o menor/maior X valor.

Só que dependendo da estrutura do pacote pode ser uma tarefa extremamente simples, então poderia focar em algo do tipo “Possibilitar que o pacote receba as formas de agregação/composição como parametro”. O Gabriel pode dar mais detalhes se seria algo factivel.

Neste notebook está um exemplo do funcionamento dos métodos que serão implemantados na smosaic.

## Download dos dados

In [ ]:
import pystac_client
import os
import sys
import pyproj
import numpy as np
import xarray as xr
import rioxarray #Funcionamento da .rio na xarray
import rasterio as rio
from pyproj import Transformer
from rasterio.enums import Resampling

In [ ]:
URL = 'https://data.inpe.br/bdc/stac/v1/'
service = pystac_client.Client.open(URL)

## Import data

In [ ]:
bbox_emas = [-53.0934, -18.1327, -52.9314, -17.9229] #wps84 Parque das emas 2026-04-01/2026-05-07
bbox_paulao=[-49.87930174,-20.37185787,-49.53239264,-20.03585787] # Região em SP periodo de interesse 2024-08-18/2024-08-25
lista = service.search(
    collections=["S2_L2A-1"],
    bbox=bbox_emas, # Recorte de interesse 
    datetime="2026-04-01/2026-05-07",
)

In [ ]:
itens = lista.item_collection()
bandas = ['B12', 'B08', 'B04', 'SCL']
links_das_imagens = []


if len(itens) == 0:
    # Não há o que processar.
    raise ValueError("Busca no STAC retornou vazia.")
else:
    for item in itens:
        links = {banda: item.assets[banda].href for banda in bandas if banda in item.assets}
        links['data'] = item.datetime.strftime('%Y-%m-%d')
        links_das_imagens.append(links)
# links_das_imagens    

In [ ]:
import odc.stac
crs_original=32722
cubo = odc.stac.load(
   itens,
    bands=['B12', 'B08', 'B04', 'SCL'],
    crs=f"EPSG:{crs_original}",  
    
    # 1. MUDANÇA DA RESOLUÇÃO ALVO
    resolution=20,     # O grid final alvo agora será de 20x20m
    
    # 2. A NOVA REGRA DE REAMOSTRAGEM (DOWNSCALING)
    resampling={
        "SCL": "nearest",    # Máscaras de nuvem
        "B04": "average",    # Agrupa 4 pixels de 10m tirando a média física para resolução de 20m
        "B08": "average",    # Agrupa 4 pixels de 10m tirando a média física para resolução de 20m
        "*": "nearest"       # A B12 já é 20m nativa, então apenas se encaixa no grid
    },
    
    # 3. Testanto paralização do processo
    chunks={'x': 1024, 'y': 1024, 'time': 1} # Ativa o Dask para não estourar a memória RAM
)

if len(cubo.time) < 1: 
    raise ValueError("Cubo carregado tem menos datas do que o necessário.")

In [ ]:
print(cubo)

In [ ]:
## Import do SMOSAIC
CLOUD_CONFIG = {
    'S2-16D-2': {
        'cloud_band': 'SCL',
        'non_cloud_values': [4, 5, 6],
        'cloud_values': [0, 1, 2, 3, 7, 8, 9, 10, 11],
        'no_data_value': 0
    },
    'S2_L2A-1': {
        'cloud_band': 'SCL',
        'non_cloud_values': [4, 5, 6],
        'cloud_values': [0, 1, 2, 3, 7, 8, 9, 10, 11],
        'no_data_value': 0
    },
    'S2_L1C_BUNDLE-1': {
        'cloud_band': 'FMASK',
        'non_cloud_values': [0, 1],
        'cloud_values': [2, 3, 4, 255],
        'no_data_value': 255
    }
}


### Aplicando mascara de nuvem

In [ ]:
#selecionando não nuvens na mascara
mask_cloud = cubo['SCL'].isin(CLOUD_CONFIG['S2-16D-2']['non_cloud_values'])
#aplicando mascara
cubo_limpo = cubo.where(mask_cloud)

### Métodos 

In [ ]:
#Mean
def gerar_composicao_media(cubo_limpo):
    """
    Retorna a composição baseada na média temporal.
    Não possui proveniência (pois a média mistura as datas).
    """
    return cubo_limpo.mean(dim='time')

In [ ]:
#Mediana
def gerar_composicao_mediana(cubo_limpo):
    """
    Retrona a composição baseada na mediana temporal.
    Não possui proveniência (pois a média mistura as datas).
    """
    return cubo_limpo.median(dim='time')

In [ ]:
#MAX
def gerar_composicao_max(cubo_limpo, cubo_original, bandas=['B12', 'B08', 'B04']):
    """
    Retorna a composição temporal onde CADA banda possui seu próprio máximo independente,
    junto com um cubo de proveniência (datas independentes para cada banda).
    """

    # Dicionários para guardar as bandas separadas antes de juntar
    dict_bandas = {}
    dict_provs = {}

    for banda in bandas:
        # 1. Blindagem contra nuvens 
        banda_temp = cubo_limpo[banda].fillna(-np.inf)
        
        # 2. Busca o índice para ESTA banda específica
        indice_max = banda_temp.argmax(dim='time').compute()

        # 3. Recorta APENAS esta banda do cubo limpo usando o seu próprio índice
        compo_max_banda = cubo_limpo[banda].isel(time=indice_max)
        
        # 4. Busca a proveniência exata em que esse máximo ocorreu
        prov_banda = cubo_original['time'].isel(time=indice_max)

        #Evitar conflito de datas
        if 'time' in compo_max_banda.coords:
            compo_max_banda = compo_max_banda.drop_vars('time')
        if 'time' in prov_banda.coords:
            prov_banda = prov_banda.drop_vars('time')

        # 5. Guarda nos dicionários com os nomes corretos
        dict_bandas[banda] = compo_max_banda
        # Nomeamos a proveniência (ex: prov_B12)
        dict_provs[f"prov_{banda}"] = prov_banda

    # 6. REMONTA A CAIXA (Dataset)! 
    dataset_maximos = xr.Dataset(dict_bandas)
    dataset_provs = xr.Dataset(dict_provs)
   
    return dataset_maximos, dataset_provs


In [ ]:
#MIN
def gerar_composicao_min(cubo_limpo, cubo_original, bandas=['B12', 'B08', 'B04']):
    """
    Retorna a composição temporal onde CADA banda possui seu próprio mínimo independente,
    junto com um cubo de proveniência (datas independentes para cada banda).
    """
    
    dict_bandas = {}
    dict_provs = {}

    for banda in bandas:
        # 1. Blindagem contra nuvens (O truque do infinito)
        banda_temp = cubo_limpo[banda].fillna(np.inf)
        
        # 2. Busca o índice para ESTA banda específica
        indice_min = banda_temp.argmin(dim='time').compute()

        # 3. Recorta APENAS esta banda do cubo limpo usando o seu próprio índice
        compo_min_banda = cubo_limpo[banda].isel(time=indice_min)
        
        # 4. Busca a proveniência exata em que esse máximo ocorreu
        prov_banda = cubo_original['time'].isel(time=indice_min)

        if 'time' in compo_min_banda.coords:
            compo_min_banda = compo_min_banda.drop_vars('time')
        if 'time' in prov_banda.coords:
            prov_banda = prov_banda.drop_vars('time')

        # 5. Guarda nos dicionários com os nomes corretos
        dict_bandas[banda] = compo_min_banda
        # Nomeamos a proveniência (ex: prov_B12)
        dict_provs[f"prov_{banda}"] = prov_banda

    # 6. REMONTA A CAIXA (Dataset)! 
    dataset_minimos = xr.Dataset(dict_bandas)
    dataset_provs = xr.Dataset(dict_provs)

    return dataset_minimos, dataset_provs


In [ ]:
def calcular_nbr_cubo(cubo):
    """
    Calcula o NBR para todas as datas do cubo simultaneamente
    e adiciona a nova banda ao Dataset.
    """
    
    # Isola as bandas inteiras (com todas as datas dentro delas)
    nir = cubo['B08']
    swir = cubo['B12']
    
    # A MÁGICA: O Xarray aplica essa conta em todos os dias e pixels de uma vez!
    # O seu epsilon (0.0000001) para evitar divisão por zero é uma ótima prática.
    numerador = nir - swir
    denominador = nir + swir + 0.0000001
    
    nbr_calculado = numerador / denominador
    
    # Adicionamos o NBR como uma nova "gaveta" dentro do cubo original
    cubo['NBR'] = nbr_calculado
    
    return cubo

In [ ]:
def achar_indice_do_enesimo(matriz_3d, enesimo, tipo='max'):
    """
    matriz_3d tem o formato (Tempo, Y, X)
    """
    quantidade_tempo = matriz_3d.shape[-1] #O tempo vai para último eixo no apply_ufunc
    
    if quantidade_tempo < 1:
        raise ValueError("Erro: A matriz temporal está vazia (0 datas). Não é possível buscar máximos ou mínimos.")

    # Garante que não vamos pedir um índice maior do que a quantidade de imagens.
    n_seguro = min(enesimo, quantidade_tempo)
    
    if tipo == 'max':
        # TRATAMENTO DE NUVENS PARA O MÁXIMO
        # Substituímos NaNs por -infinito temporariamente.
        # Assim, as nuvens vão para o início da fila na ordenação ascendente,
        # garantindo que o final da fila tenha apenas os MAIORES valores válidos.
        matriz_segura = np.where(np.isnan(matriz_3d), -np.inf, matriz_3d)
        indices_ordenados = np.argsort(matriz_segura, axis=-1)
        
        # O 1º máximo está em -1, o 2º em -2.
        return indices_ordenados[:, :, -n_seguro]
        
    elif tipo == 'min':
        # TRATAMENTO DE NUVENS PARA O MÍNIMO
        # Substituímos NaNs por +infinito temporariamente.
        # Assim, as nuvens vão para o final da fila no ordenação ascendente,
        # garantindo que o início da fila tenha apenas os MENORES valores válidos.
        matriz_segura = np.where(np.isnan(matriz_3d), np.inf, matriz_3d)
        indices_ordenados = np.argsort(matriz_segura, axis=-1)
        
        # O 1º mínimo está na posição 0, o 2º na 1. Então subtraímos 1.
        return indices_ordenados[:, :, n_seguro - 1] 
    else:
        raise ValueError("O parâmetro 'tipo' deve ser 'max' ou 'min'.")  

In [ ]:
def gerar_composicao_extrema(cubo_limpo, banda_ref='B12', enesimo=1, tipo='max'):
    """
    Retorna a composição e a proveniência baseada no enésimo máximo ou mínimo.
    """
    print(f"⏳ Calculando composição: {enesimo}º {tipo.upper()} usando {banda_ref} como referência...")
    
    # 1. Prepara o cubo para o numpy enxergar o tempo inteiro
    cubo_proc = cubo_limpo.chunk({'time': -1})
    
    # 2. Injeta a função de busca de índice
    indice_virtual = xr.apply_ufunc(
        achar_indice_do_enesimo,
        cubo_proc[banda_ref],
        kwargs={'enesimo': enesimo, 'tipo': tipo},
        input_core_dims=[['time']],
        output_core_dims=[[]],
        dask='parallelized',
        output_dtypes=[int]
    )
    
    # 3. Executa o cálculo do índice na memória
    indice_real = indice_virtual.compute()
    
    # 4. Recorta o cubo original para gerar a imagem final e as datas
    compo_final = cubo_limpo.isel(time=indice_real)
    proveniencia = cubo_limpo['time'].isel(time=indice_real)
    
    # Limpeza de dimensões mortas
    if 'time' in compo_final.coords:
        compo_final = compo_final.drop_vars('time')
        
    return compo_final, proveniencia

### Salvar em tiff

In [ ]:
def exportar_raster_radiometrico(matriz_xarray, nome_arquivo, crs_original,crs_ref=None):
    """Salva a composição de refletância (valores float) em GeoTIFF."""
    print(f"💾 Salvando imagem em: {nome_arquivo} ...")
    
    ordem_desejada = ['B12', 'B08', 'B04'] 
    bandas_presentes = list(matriz_xarray.data_vars)

    if all(b in bandas_presentes for b in ordem_desejada):
        matriz_xarray = matriz_xarray[ordem_desejada]
    else:
        print(f" -> Imagem de índice/banda única detectada ({bandas_presentes}). Pulando reordenação RGB.")

    # Etiqueta / Reprojeção -- Garante que o CRS esteja presente
    if crs_ref is not None:
        crs_atual = matriz_xarray.rio.crs
        # Verifica o CRS original (se ele não souber de onde está partindo, não consegue calcular para onde vai)
        if crs_atual is None:
             print("Aviso: O Xarray perdeu a etiqueta do CRS original")
             matriz_xarray = matriz_xarray.rio.write_crs(crs_original)
             crs_atual = matriz_xarray.rio.crs # Atualizamos a variável para o código abaixo funcionar
             
        # 2. CRS alvo diferente do ATUAL
        if str(crs_atual.to_epsg()) != str(crs_ref):
             print(f"🔄 Reprojetando de EPSG:{crs_atual.to_epsg()} para EPSG:{crs_ref} ...")
             matriz_xarray = matriz_xarray.rio.reproject(
                 f"EPSG:{crs_ref}", 
                 resampling=Resampling.bilinear # Importante para bandas de reflectância
             )
        else:
             # Se forem iguais, apenas garantimos a etiqueta
             matriz_xarray = matriz_xarray.rio.write_crs(crs_ref)

    # saber onde é null em cada banda
    for banda in matriz_xarray.data_vars:
        matriz_xarray[banda] = matriz_xarray[banda].rio.write_nodata(np.nan)
    
    # Exporta
    matriz_xarray.rio.to_raster(nome_arquivo, compress="lzw", tiled=True)
    print("✅ Raster salvo com sucesso!")

In [ ]:
def exportar_raster_proveniencia(matriz_datas, nome_arquivo, crs_original,crs_ref=None):
    """Salva a proveniência convertendo datetime64 para inteiro (AAAAMMDD)."""
    
    if isinstance(matriz_datas, xr.DataArray):
        # Damos um nome genérico se não tiver nome
        matriz_datas = matriz_datas.to_dataset(name="prov_data")

    # Lógica de conversão de data para inteiro
    datas_convertidas = {}
    
    for banda in matriz_datas.data_vars:
        # Extrai ano, mês e dia individualmente para cada banda
        datas_numericas = (
            matriz_datas[banda].dt.year * 10000 + 
            matriz_datas[banda].dt.month * 100 + 
            matriz_datas[banda].dt.day
        )
        datas_convertidas[banda] = datas_numericas.fillna(0).astype('int32')
        
    # Remonta o Dataset já como inteiros
    dataset_final = xr.Dataset(datas_convertidas)
    
    # Etiqueta / Reprojeção
    if crs_ref is not None:
        crs_atual = dataset_final.rio.crs
        
        # O xarray sofreu amnésia?
        if crs_atual is None:
             print(f"  -> Aviso: Xarray perdeu a etiqueta. Restaurando para {crs_original}...")
             dataset_final = dataset_final.rio.write_crs(crs_original)
             crs_atual = dataset_final.rio.crs 
             
        # Faz a reprojeção se for diferente
        if str(crs_atual.to_epsg()) != str(crs_ref):
             print(f"  🔄 Reprojetando proveniência de EPSG:{crs_atual.to_epsg()} para EPSG:{crs_ref} ...")
             dataset_final = dataset_final.rio.reproject(
                 f"EPSG:{crs_ref}", 
                 resampling=Resampling.nearest # 🚨 NEAREST é obrigatório para datas!
             )
        else:
             dataset_final = dataset_final.rio.write_crs(crs_ref)

    # Aplica o Nodada em todas as fitas
    for banda in dataset_final.data_vars:
        dataset_final[banda] = dataset_final[banda].rio.write_nodata(0)
        
    # Exporta
    dataset_final.rio.to_raster(nome_arquivo, compress="lzw", tiled=True)
    print(f"✅ Proveniência salva com sucesso: {nome_arquivo}")

## MAIN

### Save originais

In [ ]:
quantidade_de_datas = len(cubo.time)
print(f"Iniciando a exportação de {quantidade_de_datas} imagens originais...")

for i in range(quantidade_de_datas):
    
    # Seleção de imagem
    imagem_do_dia = cubo.isel(time=i)
    
    # Pega a data dessa imagem e formata como Texto (ex: '2023-05-12')
    # Usamos o .dt.strftime para formatar lindamente a data
    data_texto = str(imagem_do_dia.time.dt.strftime('%Y-%m-%d').values)

    nome_arquivo = f"imagem_original_{data_texto}_Paulo2025.tif"    
    print(f"Processando data {i+1}/{quantidade_de_datas}: {data_texto}")
    
    #Remove a dimensão de tempo (O rioxarray exige que o raster seja apenas Y e X)
    imagem_do_dia = imagem_do_dia.drop_vars('time')
    
    exportar_raster_radiometrico(imagem_do_dia, nome_arquivo, crs_original, 4326)

print("Todas as imagens originais exportadas!")

### Save Process

In [ ]:
# # Add NBR no cubo 
cubo_limpo = calcular_nbr_cubo(cubo_limpo)

In [ ]:
# # Produto NBR MIN
compo_max_nbr, prov_max_nbr = gerar_composicao_max(cubo_limpo, cubo, bandas=['NBR'])
exportar_raster_radiometrico(compo_max_nbr, "max_queimada_nbr_emas.tif", crs_original,4326)
exportar_raster_proveniencia(prov_max_nbr, "proveniencia_NBR_MAX_emas.tif", crs_original, 4326)

In [ ]:
# # Produto NBR MAX
compo_min_nbr, prov_min_nbr = gerar_composicao_min(cubo_limpo, cubo, bandas=['NBR'])
exportar_raster_radiometrico(compo_min_nbr, "min_queimada_nbr_emas.tif", crs_original,4326)
exportar_raster_proveniencia(prov_min_nbr, "proveniencia_NBR_MIN_emas.tif", crs_original,4326)

In [ ]:

# # # PRODUTO Média Tradicional
compo_media = gerar_composicao_media(cubo_limpo)
exportar_raster_radiometrico(compo_media, "resultado_media_emas.tif",crs_original, 4326)

In [ ]:
# # #PRODUTO Mediana 
compo_mediana = gerar_composicao_mediana(cubo_limpo)
exportar_raster_radiometrico(compo_mediana, "resultado_mediana_emas.tif", crs_original,4326)

In [ ]:
# Produto MIN do CUBO
compo_miin, prov_miin = gerar_composicao_min(cubo_limpo, cubo, bandas=['B12','B08', 'B04'])
exportar_raster_radiometrico(compo_miin, "min_queimada_emas.tif", crs_original,4326)
exportar_raster_proveniencia(prov_miin, "proveniencia_MIN_emas.tif", crs_original,4326)


In [ ]:
# Produto MAX do CUBO
compo_maax, prov_maax = gerar_composicao_min(cubo_limpo, cubo, bandas=['B12','B08', 'B04'])
exportar_raster_radiometrico(compo_maax, "max_queimada_emas.tif", crs_original,4326)
exportar_raster_proveniencia(prov_maax, "proveniencia_MAX_emas.tif", crs_original,4326)

In [ ]:
# # PRODUTO  1º Máximo  da banda de referência B08
compo_max, prov_max = gerar_composicao_extrema(cubo_limpo, banda_ref='B08', enesimo=1, tipo='max')
exportar_raster_radiometrico(compo_max, "resultado_maximo_1_B08_emas.tif", crs_original,4326)
exportar_raster_proveniencia(prov_max, "proveniencia_maximo_1_B08_emas.tif", crs_original,4326)

In [ ]:
# # PRODUTO  1º Mínimo da banda de referência B08
compo_min, prov_min = gerar_composicao_extrema(cubo_limpo, banda_ref='B08', enesimo=1, tipo='min')
exportar_raster_radiometrico(compo_min, "resultado_minimo_1_B08_emas.tif", crs_original,4326)
exportar_raster_proveniencia(prov_min, "proveniencia_minimo_1_B08_emas.tif", crs_original,4326)

In [ ]:
# # PRODUTO  2º Máximo da banda de referência B08
compo_max, prov_max = gerar_composicao_extrema(cubo_limpo, banda_ref='B08', enesimo=2, tipo='max')
exportar_raster_radiometrico(compo_max, "resultado_maximo_2_B08_emas.tif", crs_original,4326)
exportar_raster_proveniencia(prov_max, "proveniencia_maximo_2_B08_emas.tif", crs_original,4326)

In [ ]:
# # PRODUTO 2º Mínimo  da banda de referência B08
compo_min, prov_min = gerar_composicao_extrema(cubo_limpo, banda_ref='B08', enesimo=2, tipo='min')
exportar_raster_radiometrico(compo_min, "resultado_minimo_2_B08_emas.tif", crs_original,4326)
exportar_raster_proveniencia(prov_min, "proveniencia_minimo_2_B08_emas.tif",crs_original, 4326)